In [ ]:

"""
This script processes downloaded HTML files from StatCan, extracts article information 
such as the publication date, article title, and main content (including headers, paragraphs, and list items), 
and saves the cleaned text into .txt files. It also removes tables and charts. 
Files that encounter errors during processing are logged so that they can be reviewed.
"""
import requests
from bs4 import BeautifulSoup
import os
from datetime import datetime

html_dir = "html_files"
output_dir = "scraped_articles"
error_files = []

html_files = [f for f in os.listdir(html_dir) if f.endswith(".html")]


for html_file in html_files:
    file_path = os.path.join(html_dir, html_file)
    output_file_path = os.path.join(output_dir, f"{os.path.splitext(html_file)[0]}.txt")

    if os.path.exists(output_file_path):
        print(f"{output_file_path} already exists. Skipping this file.")
        continue

    try:
        with open(file_path, "r", encoding="utf-8") as f:
            page_content = f.read()

        soup = BeautifulSoup(page_content, "html.parser")

        # Extract date
        date = soup.find(class_="cite-data")
        if date:
            date_value = date.get("data-period")
            if date_value:
                try:
                    dt = datetime.strptime(date_value, "%d %B %Y")
                    formatted_date = dt.strftime("%Y-%m-%d")
                    date = formatted_date
                except ValueError:
                    date = 'Data format is incorrect'
            else:
                date = "Date Not found in the data-period attribute"
        else:
            date = "No cite-data class found"

        # Find the main section
        main = soup.find("main")
        if main:
            table_divs = main.find_all("div", class_="exportable-element simple-table")
            for table_div in table_divs:
                table_div.decompose() 

            small_tables = main.find_all("div", class_="table-scrolling-wrapper")
            for tables in small_tables:
                tables.decompose()

            # Get the header 
            header = main.find("h1").get_text(strip=True)

            # Find all content
            content = main.find_all("div", class_="abs-content clearfix text-formatted field field--name-field-abs-text-paragraph-content field--type-text-long field--label-hidden")

            # Initialize a list to collect content
            full_text_content = []

            # Iterate over each section in content
            for idx, section in enumerate(content):
                headers = section.find_all(["h2", "h3", "h4"])
                for header_tag in headers:
                    full_text_content.append(f"Header {idx+1}: {header_tag.get_text(strip=True)}")
                paragraphs = section.find_all("p")
                for p in paragraphs:
                    full_text_content.append(p.get_text(strip=True))

                # Find all list items (li) in this section
                list_items = section.find_all("li")
                for li in list_items:
                    full_text_content.append(f"- {li.get_text(strip=True)}")

            # Combine all the text into a single string
            final_text = "\n".join(full_text_content)

            # Save the text content to a file with UTF-8 encoding
            with open(output_file_path, "w", encoding="utf-8") as out_file:
                out_file.write(f"Date:\n{date}\n")
                out_file.write(f"Article Title:\n{header}\n")
                out_file.write(f"\nArticle Content:\n{final_text}\n")

            print(f"File has been saved at {output_file_path}")

    except Exception as e:
        print(f"Error processing {html_file}: {e}")
        error_files.append(html_file)

# Log files with errors
if error_files:
    error_file_path = os.path.join(output_dir, "error_files.txt")
    with open(error_file_path, "w") as error_log:
        for error_file in error_files:
            error_log.write(f"{error_file}\n")
    print(f"\nFiles with issues were logged in: {error_file_path}")
else:
    print("\nNo issues found in the processed files.")


scraped_articles/snapshot-vic-2021.txt already exists. Skipping this file.
scraped_articles/charts-casual-employment-occupation-industry-and-job-mobility-may-2023.txt already exists. Skipping this file.
scraped_articles/australia-records-another-current-account-surplus-1.txt already exists. Skipping this file.
scraped_articles/classifying-covid-19-policy-interventions-macroeconomic-statistics.txt already exists. Skipping this file.
scraped_articles/wages-rise-05-september-quarter-2019.txt already exists. Skipping this file.
scraped_articles/modelled-indicative-state-and-territory-level-industry-jobs-and-hours-worked-estimates.txt already exists. Skipping this file.
scraped_articles/10-characteristics-australian-businesses-year-ended-30-june-2022.txt already exists. Skipping this file.
scraped_articles/deaths-due-covid-19-influenza-and-rsv-australia-2022-september-2024.txt already exists. Skipping this file.
scraped_articles/ten-facts-australian-economy-september-quarter-2024.txt alread